In [3]:
# 把RAG的流程固化为两步：
# 1. Retrieve：检索知识库，返回知识片段
# 2. Generate：增强生成，基于知识片段增强Prompt和用户问题，调用模型生成答案

# 在LangChain中，提供了一个@dynamic_prompt装饰器，用来实现动态修改提示词。

from langchain.agents.middleware import dynamic_prompt, ModelRequest
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()
from langchain.agents import create_agent
from langchain_core.vectorstores import InMemoryVectorStore
#VectorStore的用法
from langchain_text_splitters import MarkdownHeaderTextSplitter

#准备数据
markdown_content = open("resources/评估.md", "r").read()

#切分文本
headers_to_split = [
    ("#","Header1"),
    ("##","Header2"),
    ("###","Header3"),
]
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split,
    strip_headers=False # 不移除标题
    )

#切分文档, 返回的是 list[Document]
docs = markdown_splitter.split_text(markdown_content)

#给vectorstore添加id
for i, doc in enumerate(docs,start=1):
    doc.id = f"doc_{i}"

#初始化VectorStore
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings

#创建ollama模型
embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b",dimensions=1024)

#初始化向量数据库
vectorstore = InMemoryVectorStore(
    embedding=embeddings
)

# 保存文本到向量数据库,返回文档id
ids = vectorstore.add_documents(docs)




#1.定义动态提示词的中间件
@dynamic_prompt
def rag_prompt(request: ModelRequest):
    #1.拿到用户问题，也就是消息列表中的最后一条消息
    print("---------用户问题：")
    user_message = request.state["messages"][-1].content
    print(user_message,"\n")

    #2.检索只是片段，list[Document]，拼接为字符串
    docs = vectorstore.similarity_search(user_message, k=2)
    context = "\n\n".join([doc.page_content for doc in docs])
    print("---------context：")
    print(context,"\n")

    return f"""你是一个教资考试专家，你根据上下文回答用户问题，不能自己编造答案。答案要简短，直接引用原文。
    Context: {context}"""

#2.初始化模型
model = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

#3.创建agent
agent = create_agent(
            model= model,
            middleware=[rag_prompt],
        )


#4.测试
from langchain.messages import HumanMessage
# 测试
response = agent.invoke({"messages":[HumanMessage("论语中教育的目的是什么？")]})
print("==============AI Message================")
print(response["messages"][-1].text)

---------用户问题：
论语中教育的目的是什么？ 

---------context：
### （三）孔子及《论语》主要思想  
**教育作用**：庶、富、教；性相近，习相远。  
**教育对象**：“有教无类”，教育民主思想。  
**教育目的**：以完善人格为教育的首要目的，培养士和君子。  
**教学内容**：文、行、忠、义。  
**教学过程**：学、思、习、行。  
**教学原则**：因材施教、启发诱导、学思结合、谦虚笃实。  
**教师观**：其身正，不令而行；其身不正，虽令不从。

### （四）孟子主要思想  
**人性论**：人性本善，人先天具有仁、义、礼、智四个“善端”。  
**教育作用**：发扬善端，培养道德完人，得天下英才而教育之。  
**教学原则**：循序渐进，专心有恒。 

==============AI Message================
以完善人格为教育的首要目的，培养士和君子。
